In [ ]:
! pip install spacy folium geopy feedparser pandas numpy
! python -m spacy download en_core_web_sm

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 17.0 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached murmurhash-1.0.15-cp39-cp39-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_5_x86_64.whl.metadata (2.3 kB)
  Using cached cymem-2.0.13-cp39-cp39-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (9.7 kB)
  Using cached preshed-3.0.12-cp39-cp39-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_5_x86_64.whl.metadata (2.5 kB)
  Using cached thinc-8.3.9-cp39-cp39-linux_x86_64.whl
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached srsly-2.5.2-cp39-cp39-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (19 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached blis-1.3.3-cp39-cp3

In [ ]:
import re
import time
from datetime import datetime, timezone

import feedparser
import folium
import pandas as pd
import spacy

from folium.plugins import HeatMap, MarkerCluster
from geopy.geocoders import Nominatim

# -------------------------------------------------
# LOAD AI MODEL
# -------------------------------------------------
# Official spaCy usage: load a pipeline like en_core_web_sm and use Doc.ents
nlp = spacy.load("en_core_web_sm")

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
RSS_FEEDS = [
    "https://feeds.reuters.com/reuters/worldNews",
    "https://www.aljazeera.com/xml/rss/all.xml",
    "https://rss.nytimes.com/services/xml/rss/nyt/World.xml",
    "https://www.theguardian.com/world/middleeast/rss",
    "https://www.bbc.com/news/world/middle_east/rss.xml",
]

MAP_CENTER = [26.0, 50.0]
MAP_ZOOM = 5
MAP_FILE = "conflict_map_ai.html"
INDEX_FILE = "index.html"

PRIMARY_TERMS = [
    "iran", "israel", "gaza", "lebanon", "tehran", "tel aviv", "jerusalem",
    "saudi arabia", "riyadh", "uae", "dubai", "abu dhabi", "qatar", "doha",
    "kuwait", "oman", "hormuz", "persian gulf", "red sea", "basra", "beirut"
]

ACTION_TERMS = [
    "strike", "missile", "attack", "drone", "bomb", "rocket", "raid",
    "explosion", "blast", "refinery", "pipeline", "airport", "port",
    "power plant", "substation", "military base", "naval base", "airspace",
    "terminal", "grid", "telecom tower", "dam", "tunnel", "bunker"
]

HIGH_SEVERITY = [
    "missile", "strike", "attack", "explosion", "bomb", "drone", "rocket", "blast"
]

INFRA_TERMS = [
    "refinery", "pipeline", "airport", "port", "power plant", "substation",
    "military base", "naval base", "dam", "telecom tower", "terminal", "airbase"
]

# dictionary fallback
PLACE_HINTS = {
    "Tehran": "Tehran, Iran",
    "Tel Aviv": "Tel Aviv, Israel",
    "Jerusalem": "Jerusalem, Israel",
    "Dubai": "Dubai, United Arab Emirates",
    "Abu Dhabi": "Abu Dhabi, United Arab Emirates",
    "Riyadh": "Riyadh, Saudi Arabia",
    "Doha": "Doha, Qatar",
    "Basra": "Basra, Iraq",
    "Beirut": "Beirut, Lebanon",
    "Hormuz": "Strait of Hormuz",
    "Jeddah": "Jeddah, Saudi Arabia",
    "Dammam": "Dammam, Saudi Arabia",
    "Muscat": "Muscat, Oman",
    "Baghdad": "Baghdad, Iraq",
    "Damascus": "Damascus, Syria",
    "Gaza": "Gaza",
    "Fujairah": "Fujairah, United Arab Emirates",
    "Ras Tanura": "Ras Tanura, Saudi Arabia",
    "Abqaiq": "Abqaiq, Saudi Arabia",
    "Natanz": "Natanz, Iran",
    "Fordow": "Fordow, Iran",
    "Bandar Abbas": "Bandar Abbas, Iran",
    "Bushehr": "Bushehr, Iran",
    "Kharg Island": "Kharg Island, Iran",
    "Assaluyeh": "Assaluyeh, Iran",
    "Ras Laffan": "Ras Laffan Industrial City, Qatar",
}

# manually prioritize strategic places if found
STRATEGIC_PLACE_PRIORITY = [
    "Ras Tanura", "Abqaiq", "Natanz", "Fordow", "Kharg Island", "Assaluyeh",
    "Bandar Abbas", "Bushehr", "Ras Laffan", "Dubai", "Abu Dhabi",
    "Tehran", "Tel Aviv", "Jerusalem", "Basra", "Beirut", "Doha", "Riyadh"
]

# -------------------------------------------------
# HELPERS
# -------------------------------------------------
geolocator = Nominatim(user_agent="entropymap_ai")
geo_cache = {}

def clean_text(text: str) -> str:
    text = re.sub(r"<[^>]+>", " ", text or "")
    return re.sub(r"\s+", " ", text).strip()

def geocode_location(loc: str):
    if loc in geo_cache:
        return geo_cache[loc]
    try:
        point = geolocator.geocode(loc, timeout=10)
        if point:
            geo_cache[loc] = {"lat": point.latitude, "lon": point.longitude}
            return geo_cache[loc]
    except Exception:
        pass
    return None

def is_relevant(text: str) -> bool:
    t = text.lower()
    return any(p in t for p in PRIMARY_TERMS) or any(a in t for a in ACTION_TERMS)

def severity_score(text: str) -> int:
    score = 3
    t = text.lower()
    for w in HIGH_SEVERITY:
        if w in t:
            score += 4
    for w in INFRA_TERMS:
        if w in t:
            score += 3
    score += min(len(t.split()) // 12, 3)
    return min(score, 15)

def alert_level(s: int) -> str:
    if s >= 13:
        return "Critical"
    if s >= 10:
        return "High"
    if s >= 6:
        return "Medium"
    return "Low"

def alert_fill_color(level: str) -> str:
    return {
        "Critical": "#8B0000",
        "High": "#FF4500",
        "Medium": "#FFD700",
        "Low": "#90EE90",
    }.get(level, "#D3D3D3")

def detect_side(text: str) -> str:
    t = text.lower()
    if "iran" in t and ("israel" in t or "u.s." in t or "us " in t or "american" in t):
        return "Mixed"
    if "iran" in t:
        return "Iran"
    if "israel" in t:
        return "Israel"
    if "u.s." in t or "us " in t or "american" in t or "pentagon" in t or "washington" in t:
        return "US"
    return "Other"

def border_color_for_side(side: str) -> str:
    return {
        "Iran": "red",
        "Israel": "blue",
        "US": "green",
        "Mixed": "purple",
        "Other": "gray"
    }.get(side, "gray")

def detect_infrastructure(text: str) -> str:
    t = text.lower()
    if any(k in t for k in ["refinery", "pipeline", "oil", "gas", "terminal", "port", "lng"]):
        return "Energy / Maritime"
    if any(k in t for k in ["power plant", "grid", "substation", "blackout"]):
        return "Electric Grid"
    if any(k in t for k in ["airport", "airspace", "runway", "flight", "airbase"]):
        return "Aviation"
    if any(k in t for k in ["telecom", "tower", "satellite", "data center"]):
        return "Communication"
    if any(k in t for k in ["dam", "reservoir", "desalination", "water"]):
        return "Water"
    if any(k in t for k in ["military base", "naval base", "bunker", "missile base"]):
        return "Military"
    if any(k in t for k in ["government", "ministry", "embassy", "parliament"]):
        return "Government"
    return "Other"

def deduplicate_items(items):
    seen = set()
    out = []
    for item in items:
        key = (
            item["title"].strip().lower(),
            round(item["lat"], 2),
            round(item["lon"], 2),
        )
        if key not in seen:
            seen.add(key)
            out.append(item)
    return out

# -------------------------------------------------
# AI LOCATION EXTRACTION
# -------------------------------------------------
def extract_location_dictionary(text: str):
    matches = []
    t = text.lower()
    for hint, canon in PLACE_HINTS.items():
        if hint.lower() in t:
            matches.append((hint, canon))
    matches.sort(key=lambda x: len(x[0]), reverse=True)
    return matches[0][1] if matches else None

def extract_location_ai(text: str):
    """
    Use spaCy NER to pull GPE/LOC/FAC entities.
    Then normalize strategic names through PLACE_HINTS when possible.
    """
    doc = nlp(text)
    candidates = []

    for ent in doc.ents:
        if ent.label_ in {"GPE", "LOC", "FAC"}:
            ent_text = ent.text.strip()
            if ent_text:
                candidates.append(ent_text)

    # prioritize strategic matches if any candidate contains them
    for strategic in STRATEGIC_PLACE_PRIORITY:
        for cand in candidates:
            if strategic.lower() in cand.lower() or cand.lower() in strategic.lower():
                return PLACE_HINTS.get(strategic, strategic)

    # normalize candidates against dictionary if possible
    for cand in candidates:
        for hint, canon in PLACE_HINTS.items():
            if hint.lower() == cand.lower() or hint.lower() in cand.lower() or cand.lower() in hint.lower():
                return canon

    # if AI found something but no dictionary normalization, return raw candidate
    if candidates:
        return candidates[0]

    return None

def extract_location_hybrid(text: str):
    """
    Prefer AI first, fallback to dictionary.
    """
    loc_ai = extract_location_ai(text)
    if loc_ai:
        return loc_ai, "ai"

    loc_dict = extract_location_dictionary(text)
    if loc_dict:
        return loc_dict, "dictionary"

    return None, None

def confidence_score(text: str, loc_source: str, infrastructure: str) -> float:
    score = 0.35

    t = text.lower()

    if loc_source == "ai":
        score += 0.30
    elif loc_source == "dictionary":
        score += 0.20

    if any(k in t for k in PRIMARY_TERMS):
        score += 0.15
    if any(k in t for k in ACTION_TERMS):
        score += 0.15
    if infrastructure != "Other":
        score += 0.10

    return round(min(score, 0.99), 2)

def ai_summary(title: str, location: str, infra: str, level: str) -> str:
    return f"{level} alert involving {infra.lower()} near {location}."

# -------------------------------------------------
# PIPELINE
# -------------------------------------------------
def parse_rss():
    incidents = []
    now_utc = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

    for url in RSS_FEEDS:
        try:
            feed = feedparser.parse(url)
        except Exception:
            continue

        source_title = clean_text(feed.get("feed", {}).get("title", url))
        entries = getattr(feed, "entries", [])[:20]

        for entry in entries:
            title = clean_text(getattr(entry, "title", ""))
            summary = clean_text(getattr(entry, "summary", ""))
            text = f"{title} {summary}"

            if not is_relevant(text):
                continue

            infra = detect_infrastructure(text)
            location, loc_source = extract_location_hybrid(text)
            if not location:
                continue

            geo = geocode_location(location)
            time.sleep(1)

            if not geo:
                continue

            sev = severity_score(text)
            level = alert_level(sev)
            side = detect_side(text)
            conf = confidence_score(text, loc_source, infra)
            summary_line = ai_summary(title, location, infra, level)

            incidents.append({
                "published": now_utc,
                "title": title,
                "location": location,
                "location_source": loc_source,
                "lat": geo["lat"],
                "lon": geo["lon"],
                "severity": sev,
                "alert_level": level,
                "side": side,
                "infrastructure": infra,
                "confidence": conf,
                "ai_summary": summary_line,
                "source": source_title,
                "link": getattr(entry, "link", ""),
            })

    return deduplicate_items(incidents)

# -------------------------------------------------
# MAP + DASHBOARD
# -------------------------------------------------
def build_map(incidents):
    m = folium.Map(location=MAP_CENTER, zoom_start=MAP_ZOOM, tiles="CartoDB positron")
    cluster = MarkerCluster().add_to(m)

    for i in incidents:
        folium.CircleMarker(
            location=[i["lat"], i["lon"]],
            radius=max(6, i["severity"]),
            color=border_color_for_side(i["side"]),
            weight=2,
            fill=True,
            fill_color=alert_fill_color(i["alert_level"]),
            fill_opacity=0.75,
            popup=(
                f"<b>{i['title']}</b><br>"
                f"<b>AI summary:</b> {i['ai_summary']}<br>"
                f"<b>Location:</b> {i['location']}<br>"
                f"<b>Location source:</b> {i['location_source']}<br>"
                f"<b>Side:</b> {i['side']}<br>"
                f"<b>Infrastructure:</b> {i['infrastructure']}<br>"
                f"<b>Alert:</b> {i['alert_level']}<br>"
                f"<b>Severity:</b> {i['severity']}<br>"
                f"<b>Confidence:</b> {i['confidence']}<br>"
                f"<b>Source:</b> {i['source']}<br>"
                f"<a href='{i['link']}' target='_blank'>Open article</a>"
            ),
            tooltip=f"{i['location']} | {i['alert_level']} | conf {i['confidence']}"
        ).add_to(cluster)

    if incidents:
        HeatMap([[x["lat"], x["lon"], x["severity"]] for x in incidents], radius=35, blur=20).add_to(m)

    return m

def make_dashboard(df: pd.DataFrame):
    table_html = df.to_html(index=False, escape=False)
    html = f"""<html>
<head>
<meta charset="utf-8">
<title>EntropyMap AI Dashboard</title>
<style>
body {{ font-family: Arial, sans-serif; margin: 20px; }}
h1 {{ margin-bottom: 8px; }}
.note {{ color: #555; margin-bottom: 18px; }}
iframe {{ width: 100%; height: 700px; border: 1px solid #ccc; }}
table {{ border-collapse: collapse; width: 100%; font-size: 13px; }}
th, td {{ border: 1px solid #ddd; padding: 8px; vertical-align: top; }}
th {{ background: #f2f2f2; }}
</style>
</head>
<body>
<h1>EntropyMap AI Dashboard</h1>
<div class="note">AI-enhanced location extraction via spaCy NER + dictionary fallback.</div>
<iframe src="{MAP_FILE}"></iframe>
<h2>Detected incidents</h2>
{table_html}
</body>
</html>
"""
    with open(INDEX_FILE, "w", encoding="utf-8") as f:
        f.write(html)

def main():
    incidents = parse_rss()
    df = pd.DataFrame(incidents, columns=[
        "published","location","location_source","side","infrastructure",
        "alert_level","severity","confidence","ai_summary","source","title","link"
    ])
    build_map(incidents).save(MAP_FILE)
    make_dashboard(df)
    print(f"Incidents detected: {len(incidents)}")
    print(f"Wrote {MAP_FILE} and {INDEX_FILE}")

if __name__ == "__main__":
    main()
'''

requirements = '''folium
geopy
feedparser
pandas
spacy
'''

notes = '''Add this once in your environment if you run locally:
python -m spacy download en_core_web_sm

For GitHub Actions, you should also add a workflow step:
python -m spacy download en_core_web_sm
'''

(base / "update_conflict_map_ai.py").write_text(script, encoding="utf-8")
(base / "requirements_ai.txt").write_text(requirements, encoding="utf-8")
(base / "spacy_notes.txt").write_text(notes, encoding="utf-8")

print("Created files:")
for name in ["update_conflict_map_ai.
::contentReference[oaicite:1]{index=1}
py", "requirements_ai.txt", "spacy_notes.txt"]:
    print(base / name)